# Kaggle - Regression

**Objectif :**
- Charger les données
- Vérifier les données
- Utiliser `target` comme variable cible
- Utiliser `patient_id` pour le GroupKFold
- Comparer un modèle de base et ExtraTrees
- Tuner ExtraTrees
- Créer la prédiction Kaggle
- Créer le fichier de submission

## 1. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor

In [ ]:
X = pd.read_csv("../data/X_train.csv")
y_raw = pd.read_csv("../data/y_train.csv")
X_test = pd.read_csv("../data/X_test.csv")

print("X_train :", X.shape)
print("y_train :", y_raw.shape)
print("X_test :", X_test.shape)

## 2. Vérification des données

In [ ]:
print("Colonnes de X_train :")
print(X.columns.tolist())

print("\nColonnes de y_train :")
print(y_raw.columns.tolist())

print("\nPremières lignes de X_train :")
display(X.head())

print("\nPremières lignes de y_train :")
display(y_raw.head())

print("\nIndex X_train == y_train :",
      X["Index"].equals(y_raw["Index"]))

## 3. Target

In [ ]:
# IMPORTANT :
# La cible est uniquement la colonne "target".
# La colonne "Index" ne doit pas être prédite.

y = y_raw["target"]

print(y.describe())

## 4. Valeurs manquantes

In [ ]:
missing_count = X.isna().sum()
missing_percentage = X.isna().mean() * 100

missing_table = pd.DataFrame({
    "Nombre de valeurs manquantes": missing_count,
    "Pourcentage (%)": missing_percentage.round(2)
})

missing_table = missing_table[
    missing_table["Nombre de valeurs manquantes"] > 0
]

missing_table = missing_table.sort_values(
    by="Nombre de valeurs manquantes",
    ascending=False
)

display(missing_table)

## 5. Préparation des variables

In [ ]:
# Index est un identifiant de ligne.
# patient_id sert uniquement pour GroupKFold.

X_model = X.drop(
    columns=["Index", "patient_id"]
).copy()

X_test_model = X_test.drop(
    columns=["Index", "patient_id"]
).copy()

print("Variables utilisées :")
print(X_model.columns.tolist())

In [ ]:
print(X_model.dtypes)

In [ ]:
# Conversion des variables texte si nécessaire

for column in X_model.columns:

    if X_model[column].dtype == "object":

        categories = (
            X_model[column]
            .astype("category")
            .cat.categories
        )

        mapping = {
            category: i
            for i, category in enumerate(categories)
        }

        X_model[column] = (
            X_model[column]
            .astype("category")
            .cat.codes
        )

        X_test_model[column] = (
            X_test_model[column]
            .map(mapping)
            .fillna(-1)
        )


# Valeurs infinies
X_model = X_model.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test_model = X_test_model.replace(
    [np.inf, -np.inf],
    np.nan
)


# Valeurs manquantes
# On utilise uniquement les médianes du train.

medians = X_model.median()

X_model = X_model.fillna(medians)

X_test_model = X_test_model.fillna(medians)

print("Préparation terminée.")

## 6. GroupKFold

In [ ]:
# Toutes les visites d'un même patient
# restent dans le même fold.

groups = X["patient_id"]

cv = GroupKFold(
    n_splits=5
)

print(
    "Nombre de patients :",
    groups.nunique()
)

print(
    "Nombre de folds :",
    cv.n_splits
)

## 7. Modèle de base : DummyRegressor

In [ ]:
dummy_scores      = []
dummy_rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_model, y, groups),
    start=1
):

    X_train_fold = X_model.iloc[train_idx]
    X_val_fold = X_model.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model = DummyRegressor(
        strategy="mean"
    )

    model.fit(
        X_train_fold,
        y_train_fold
    )

    predictions = model.predict(
        X_val_fold
    )

    mae  = mean_absolute_error(y_val_fold, predictions)
    rmse = root_mean_squared_error(y_val_fold, predictions)

    dummy_scores.append(mae)
    dummy_rmse_scores.append(rmse)

    print(
        f"Fold {fold} - MAE : {mae:.4f}  |  RMSE : {rmse:.4f}"
    )


print("\nMAE moyen :", np.mean(dummy_scores))
print("Ecart-type MAE :", np.std(dummy_scores))
print("RMSE moyen :", np.mean(dummy_rmse_scores))
print("Ecart-type RMSE :", np.std(dummy_rmse_scores))

## 8. ExtraTrees sans tuning

In [ ]:
model = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

scores      = []
rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_model, y, groups),
    start=1
):

    X_train_fold = X_model.iloc[train_idx]
    X_val_fold = X_model.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model.fit(
        X_train_fold,
        y_train_fold
    )

    predictions = model.predict(
        X_val_fold
    )

    mae  = mean_absolute_error(y_val_fold, predictions)
    rmse = root_mean_squared_error(y_val_fold, predictions)

    scores.append(mae)
    rmse_scores.append(rmse)

    print(
        f"Fold {fold} - MAE : {mae:.4f}  |  RMSE : {rmse:.4f}"
    )


print("\nMAE moyen :", np.mean(scores))
print("Ecart-type MAE :", np.std(scores))
print("RMSE moyen :", np.mean(rmse_scores))
print("Ecart-type RMSE :", np.std(rmse_scores))

## 9. Tuning de ExtraTrees

In [ ]:
param_grid = {

    "n_estimators": [
        200,
        400
    ],

    "max_depth": [
        None,
        15,
        25
    ],

    "min_samples_leaf": [
        1,
        2,
        4
    ],

    "max_features": [
        1.0,
        "sqrt"
    ]
}

In [ ]:
tuned_model = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(

    estimator=tuned_model,

    param_grid=param_grid,

    scoring="neg_mean_absolute_error",

    cv=cv,

    n_jobs=-1,

    verbose=1
)

grid_search.fit(
    X_model,
    y,
    groups=groups
)

print("Meilleurs paramètres :")
print(grid_search.best_params_)

print("\nMeilleur MAE CV :")
print(-grid_search.best_score_)

## 10. Vérification du modèle tuné

In [ ]:
best_model = grid_search.best_estimator_

tuned_scores      = []
tuned_rmse_scores = []

last_y_val = None
last_predictions = None

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_model, y, groups),
    start=1
):

    X_train_fold = X_model.iloc[train_idx]
    X_val_fold = X_model.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    best_model.fit(
        X_train_fold,
        y_train_fold
    )

    predictions = best_model.predict(
        X_val_fold
    )

    mae  = mean_absolute_error(y_val_fold, predictions)
    rmse = root_mean_squared_error(y_val_fold, predictions)

    tuned_scores.append(mae)
    tuned_rmse_scores.append(rmse)

    last_y_val = y_val_fold
    last_predictions = predictions

    print(
        f"Fold {fold} - MAE : {mae:.4f}  |  RMSE : {rmse:.4f}"
    )


print("\nMAE moyen :", np.mean(tuned_scores))
print("Ecart-type MAE :", np.std(tuned_scores))
print("RMSE moyen :", np.mean(tuned_rmse_scores))
print("Ecart-type RMSE :", np.std(tuned_rmse_scores))

## 11. Valeurs réelles vs prédictions

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    last_y_val,
    last_predictions,
    alpha=0.3
)

minimum = min(
    last_y_val.min(),
    last_predictions.min()
)

maximum = max(
    last_y_val.max(),
    last_predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum]
)

plt.xlabel("Valeurs réelles")
plt.ylabel("Prédictions")

plt.title(
    "Valeurs réelles vs prédictions"
)

plt.show()

## 12. Entraînement final

In [ ]:
# On entraîne le meilleur modèle
# sur toutes les données d'entraînement.

best_model.fit(
    X_model,
    y
)

test_predictions = best_model.predict(
    X_test_model
)

print(
    "Nombre de prédictions :",
    len(test_predictions)
)

print("\nPremières prédictions :")
print(test_predictions[:10])

## 13. Création de la submission

In [ ]:
submission = pd.DataFrame({

    "Index": X_test["Index"],

    "target": test_predictions
})

print(submission.head())

print(
    "\nTaille :",
    submission.shape
)

## 14. Vérifications

In [ ]:
assert len(submission) == len(X_test)

assert submission["Index"].equals(
    X_test["Index"]
)

assert submission["target"].notna().all()

print(
    "Toutes les vérifications sont OK."
)

## 15. Sauvegarde

In [ ]:
from pathlib import Path

Path("../submissions").mkdir(
    exist_ok=True
)

submission_path = (
    "../submissions/"
    "SubmitTest2.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print(
    "Fichier sauvegardé :"
)

print(submission_path)

## Résumé

- `target` est la vraie cible.
- `Index` est retiré du modèle.
- `patient_id` sert uniquement au `GroupKFold`.
- On compare avec un `DummyRegressor`.
- On utilise `ExtraTreesRegressor`.
- Les hyperparamètres sont tunés avec `GridSearchCV`.
- La métrique utilisée est le MAE.
- Le fichier final est `submission_tuned_extratrees.csv`.